# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrishaSolanki-coder/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

I will rank content items using a simple baseline score based on observable current performance signals. Pages with high impressions but relatively low sessions will receive higher priority because they have search visibility but may have an opportunity for improvement.

Reason codes:
- CTR_OPPORTUNITY: high impressions with relatively low CTR
- LOW_VOLUME: low impressions and sessions
- STRONG_PAGE: good current performance, so monitor rather than refresh

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "https://raw.githubusercontent.com/PrishaSolanki-coder/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
)

df["score"] = (
    df["impressions_90d"].rank(pct=True) * 0.5
    + (1 - df["ctr"].rank(pct=True)) * 0.3
    + (1 - df["sessions_90d"].rank(pct=True)) * 0.2
)

df["reason_code"] = np.select(
    [
        (df["impressions_90d"] > df["impressions_90d"].median()) &
        (df["ctr"] < df["ctr"].median()),
        (df["impressions_90d"] < df["impressions_90d"].median()) &
        (df["sessions_90d"] < df["sessions_90d"].median())
    ],
    [
        "CTR_OPPORTUNITY",
        "LOW_VOLUME"
    ],
    default="STRONG_PAGE"
)

df["action"] = np.select(
    [
        df["reason_code"] == "CTR_OPPORTUNITY",
        df["reason_code"] == "LOW_VOLUME"
    ],
    [
        "Refresh",
        "Monitor"
    ],
    default="Protect"
)

print(df[["score", "reason_code", "action"]].head())

      score      reason_code   action
0  0.472887      STRONG_PAGE  Protect
1  0.710722  CTR_OPPORTUNITY  Refresh
2  0.678495      STRONG_PAGE  Protect
3  0.513645      STRONG_PAGE  Protect
4  0.604498      STRONG_PAGE  Protect


The queue ranks every content item from highest to lowest baseline score. The score uses only current observable signals and does not use future outcomes or label-derived fields.

In [4]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue = df.sort_values("score", ascending=False).copy()
queue["rank"] = range(1, len(queue) + 1)

output = queue[
    ["rank", "content_id", "score", "reason_code", "action"]
]

output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows written:", len(output))
output.head(20)

Rows written: 30000


,rank,content_id,score,reason_code,action
25462,1,content_825a9788af8d,0.881343,CTR_OPPORTUNITY,Refresh
12869,2,content_5d5653c4eb4f,0.877502,CTR_OPPORTUNITY,Refresh
9443,3,content_8ba781dafa55,0.852942,CTR_OPPORTUNITY,Refresh
10136,4,content_df71843dcd17,0.852152,CTR_OPPORTUNITY,Refresh
7445,5,content_c8e9d6ab9013,0.841915,CTR_OPPORTUNITY,Refresh
6967,6,content_ae6d1339904d,0.838118,CTR_OPPORTUNITY,Refresh
23220,7,content_f986bd514b6e,0.834022,CTR_OPPORTUNITY,Refresh
4458,8,content_05133844bff4,0.831418,CTR_OPPORTUNITY,Refresh
5151,9,content_e09b5602ba42,0.828900,CTR_OPPORTUNITY,Refresh
17362,10,content_c82bc0c24241,0.828102,CTR_OPPORTUNITY,Refresh


I reviewed the top 20 ranked items using their action, reason code, and score. These are baseline decision-support recommendations, not guaranteed outcomes. A recommendation could be wrong if the current signals do not reflect the page's actual content quality or business importance.

In [5]:
top20 = queue.head(20)[
    ["rank", "content_id", "score", "reason_code", "action"]
].copy()

top20["confidence_note"] = "Baseline signal-based recommendation"
top20["what_would_make_it_wrong"] = (
    "Current signals may not represent actual content quality or business value"
)

top20

,rank,content_id,score,reason_code,action,confidence_note,what_would_make_it_wrong
25462,1,content_825a9788af8d,0.881343,CTR_OPPORTUNITY,Refresh,Baseline signal-based recommendation,Current signals may not represent actual conte...
12869,2,content_5d5653c4eb4f,0.877502,CTR_OPPORTUNITY,Refresh,Baseline signal-based recommendation,Current signals may not represent actual conte...
9443,3,content_8ba781dafa55,0.852942,CTR_OPPORTUNITY,Refresh,Baseline signal-based recommendation,Current signals may not represent actual conte...
10136,4,content_df71843dcd17,0.852152,CTR_OPPORTUNITY,Refresh,Baseline signal-based recommendation,Current signals may not represent actual conte...
7445,5,content_c8e9d6ab9013,0.841915,CTR_OPPORTUNITY,Refresh,Baseline signal-based recommendation,Current signals may not represent actual conte...
6967,6,content_ae6d1339904d,0.838118,CTR_OPPORTUNITY,Refresh,Baseline signal-based recommendation,Current signals may not represent actual conte...
23220,7,content_f986bd514b6e,0.834022,CTR_OPPORTUNITY,Refresh,Baseline signal-based recommendation,Current signals may not represent actual conte...
4458,8,content_05133844bff4,0.831418,CTR_OPPORTUNITY,Refresh,Baseline signal-based recommendation,Current signals may not represent actual conte...
5151,9,content_e09b5602ba42,0.828900,CTR_OPPORTUNITY,Refresh,Baseline signal-based recommendation,Current signals may not represent actual conte...
17362,10,content_c82bc0c24241,0.828102,CTR_OPPORTUNITY,Refresh,Baseline signal-based recommendation,Current signals may not represent actual conte...


Some weak picks may receive a high score because the baseline uses only a few signals and does not know the actual editorial context. This is a limitation of the rule. I also checked that future-window outcomes and label-derived fields such as trend_pct, trend_direction, and is_declining_label are not used in the score.

In [3]:
leakage_fields = [
    "trend_pct",
    "trend_direction",
    "is_declining_label"
]

used_fields = ["impressions_90d", "ctr", "sessions_90d"]

print("Used fields:", used_fields)
print("Leakage fields excluded:", leakage_fields)

weak_picks = queue.tail(5)[
    ["rank", "content_id", "score", "reason_code", "action"]
]

weak_picks

Used fields: ['impressions_90d', 'ctr', 'sessions_90d']
Leakage fields excluded: ['trend_pct', 'trend_direction', 'is_declining_label']


,rank,content_id,score,reason_code,action
4490,29996,content_255764fc8502,0.144795,STRONG_PAGE,Protect
803,29997,content_d2b1c270cc16,0.137080,STRONG_PAGE,Protect
12776,29998,content_cb3736ca94cc,0.134392,STRONG_PAGE,Protect
21639,29999,content_2175bfa1a0cc,0.125195,STRONG_PAGE,Protect
4526,30000,content_6d4472cbbaaa,0.124095,STRONG_PAGE,Protect


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.